In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [7]:
from langchain_classic.retrievers import MultiVectorRetriever
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.stores import InMemoryStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser

import uuid

In [4]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyMuPDFLoader("data/SPRI_AI_Brief_2023년12월호_F.pdf")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = loader.load_and_split(splitter)

In [5]:
target_docs = split_docs[:5]

summary_prompt = ChatPromptTemplate.from_template(
    """다음 문서의 핵심 내용을 검색에 적합하도록 간결하게 요약해줘.
    
    문서:
    {doc}
    """
)

summary_chain = (
    {"doc": lambda doc: doc.page_content}
    | summary_prompt
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | StrOutputParser()
)

summaries = summary_chain.batch(target_docs)

print("원본 문서 수:", len(target_docs))
print("생성된 요약 수:", len(summaries))
print("\n[첫 번째 요약]")
print(summaries[0])

원본 문서 수: 5
생성된 요약 수: 5

[첫 번째 요약]
2023년 12월호의 핵심 내용 요약: 해당 월의 주요 기사 및 주제, 트렌드, 이벤트 등을 포함한 정보 제공.


In [8]:
doc_ids = [str(uuid.uuid4()) for _ in target_docs]

summary_docs = [
    Document(
        page_content= summary,
        metadata = {"doc_id": doc_ids[i]},
    )
    for i, summary in enumerate(summaries)
]

print("요약 문서 수:", len(summary_docs))
print("\n첫 번째 요약:")
print(summary_docs[0].page_content)
print("\n첫 번째 요약의 metadata:")
print(summary_docs[0].metadata)

요약 문서 수: 5

첫 번째 요약:
2023년 12월호의 핵심 내용 요약: 해당 월의 주요 기사 및 주제, 트렌드, 이벤트 등을 포함한 정보 제공.

첫 번째 요약의 metadata:
{'doc_id': 'caf416d1-47b7-4fdc-ada6-bc7544e2deac'}


In [ ]:
vectorstore = Chroma(
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
)

vectorstore.add_documents(summary_docs)

# 원본 저장
store = InMemoryStore()
store.mset(list(zip(doc_ids, target_docs)))

print("Vector Store에 저장한 요약 수:", len(summary_docs))
print("원본 저장소에 저장한 문서 수:", len(list(store.yield_keys())))

In [14]:
from langchain_classic.retrievers import MultiVectorRetriever

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key="doc_id"
)

In [19]:
query = "AI 기술의 최근 동향을 알려줘."

summary_results = vectorstore.similarity_search(query, k=1)

oriainal_results = retriever.invoke(query)

print("질문")
print(query)

for i, doc in enumerate(summary_results, start=1):
    print("요약")
    print(doc.page_content)
    print(doc.metadata)
    
for i, doc in enumerate(oriainal_results, start=1):
    print("원본")
    print(doc.page_content)
    print(doc.metadata)
    

질문
AI 기술의 최근 동향을 알려줘.
요약
- EU AI 법 3자 협상에서 기반모델 규제에 대한 견해 차이로 진전이 어려움.
- 미국 프런티어 모델 포럼, 1,000만 달러 규모의 AI 안전 기금 조성.
- 코히어, 데이터 투명성을 위한 데이터 출처 탐색기 공개.
- 알리바바 클라우드, 최신 LLM ‘통이치엔원 2.0’ 발표.
- 삼성전자, 자체 개발한 생성 AI ‘삼성 가우스’ 공개.
{'doc_id': '61ddcdcb-7157-4115-bc85-28bf37b2cc53'}
원본
▹ EU AI 법 3자 협상, 기반모델 규제 관련 견해차로 난항··················································· 6
 
 2. 기업/산업 
   ▹ 미국 프런티어 모델 포럼, 1,000만 달러 규모의 AI 안전 기금 조성································ 7
   ▹ 코히어, 데이터 투명성 확보를 위한 데이터 출처 탐색기 공개  ······································· 8
   ▹ 알리바바 클라우드, 최신 LLM ‘통이치엔원 2.0’ 공개 ······················································ 9
   ▹ 삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개 ··························································· 10
{'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2018 10.0.0.13462', 'creationdate': '2023-12-08T13:28:38+09:00', 'source': 'data/SPRI_AI_Brief_2023년12월호_F.pdf', 'file_path': 'data/SPRI_AI_Brief_2023년12월호_F.pdf', 'total_pages': 23, 'format': 'PDF 1.4', 'title': '', 